# Transferred-teacher integrity rerun
This notebook keeps the official LOL-v2 Real test set untouched until the final stage. Upload this complete rerun folder as one Kaggle dataset, add LOL-v2, and add the exact revised `teacher_illum_ft.pth` checkpoint dataset. Despite its legacy filename/metadata, the checkpoint is verified below as the actual six-channel model.

In [ ]:
!pip install -q lpips scikit-image

from pathlib import Path
import os, shutil, subprocess, sys, zipfile, hashlib

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/Transferred_Teacher_Integrity')

source_candidates = [p for p in INPUT.rglob('src/train_distill.py') if (p.parents[1] / 'scripts/run_integrity_protocol.py').is_file()]
if not source_candidates:
    for archive in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive) as zf:
                names = zf.namelist()
                if (any(name.endswith('src/train_distill.py') for name in names) and
                    any(name.endswith('scripts/run_integrity_protocol.py') for name in names)):
                    zf.extractall('/kaggle/working/uploaded_integrity')
                    break
        except zipfile.BadZipFile:
            pass
    source_candidates = [p for p in Path('/kaggle/working/uploaded_integrity').rglob('src/train_distill.py') if (p.parents[1] / 'scripts/run_integrity_protocol.py').is_file()]
if not source_candidates:
    raise FileNotFoundError('The transferred-teacher integrity package was not found.')
SOURCE_ROOT = source_candidates[0].parents[1]
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SOURCE_ROOT, WORK)

dataset_candidates = list(INPUT.rglob('LOL-v2/Real_captured/Train/Low'))
if not dataset_candidates:
    dataset_candidates = list(INPUT.rglob('Real_captured/Train/Low'))
if not dataset_candidates:
    raise FileNotFoundError('LOL-v2 Real training folders were not found.')
DATASET_ROOT = dataset_candidates[0].parents[2]

checkpoint_candidates = [p for p in INPUT.rglob('teacher_illum_ft.pth') if p.is_file()]
if not checkpoint_candidates:
    raise FileNotFoundError('teacher_illum_ft.pth was not found.')
if len(checkpoint_candidates) == 1:
    INIT_CHECKPOINT = checkpoint_candidates[0]
else:
    print('Multiple teacher_illum_ft.pth files found:')
    for i, path in enumerate(checkpoint_candidates): print(i, path)
    raise RuntimeError('Attach only the exact revised teacher checkpoint, then rerun this cell.')
EXPECTED_TEACHER_SHA256 = 'a92c2988ced5cb0fb8e403fb143913d8fc2837fdb31dbeb1634aa3ca031697a2'
actual_hash = hashlib.sha256(INIT_CHECKPOINT.read_bytes()).hexdigest()
if actual_hash != EXPECTED_TEACHER_SHA256:
    raise RuntimeError(f'Wrong teacher checkpoint SHA-256: {actual_hash}')
print('Integrity source:', SOURCE_ROOT)
print('Working copy:', WORK)
print('LOL-v2 root:', DATASET_ROOT)
print('Transferred teacher:', INIT_CHECKPOINT)
print('Teacher SHA-256:', actual_hash)

In [ ]:
# Configuration. Use 3407,3408,3409 for the strongest evidence; one seed is a faster first pass.
SEEDS = '3407'
EPOCHS = 20
RUNNER = WORK / 'scripts/run_integrity_protocol.py'
COMMON = [
    sys.executable, str(RUNNER),
    '--dataset-root', str(DATASET_ROOT),
    '--init-checkpoint', str(INIT_CHECKPOINT),
    '--seeds', SEEDS,
    '--epochs', str(EPOCHS),
]
subprocess.run(COMMON + ['--stage', 'split'], check=True)
subprocess.run([sys.executable, str(WORK / 'tests/test_integrity.py')], check=True)

## Stage 1: controlled training
For every seed this runs two matched branches: anchor-only continued training and FSD plus the identical anchor. Both start from the same checkpoint and use the same split, crops, epochs, learning rate, sampled states, and seed.

In [ ]:
subprocess.run(COMMON + ['--stage', 'train'], check=True)

## Stage 2: ARR selection on validation only
All alpha values use the same per-image starting latents. No official test image is evaluated in this stage.

In [ ]:
subprocess.run(COMMON + ['--stage', 'tune'], check=True)
import pandas as pd
display(pd.read_csv(WORK / 'results/arr_validation_aggregate.csv'))
print((WORK / 'results/selected_alpha.json').read_text())

## Stage 3: frozen official test evaluation
Run this only after Stage 2 has selected alpha. Do not return to Stage 2 based on these test results.

In [ ]:
subprocess.run(COMMON + ['--stage', 'official-test'], check=True)
subprocess.run([sys.executable, str(WORK / 'scripts/summarize_results.py'), str(WORK / 'results/official_test_registry.csv'), '--output-dir', str(WORK / 'results/summary')], check=True)
display(pd.read_csv(WORK / 'results/official_test_registry.csv'))
display(pd.read_csv(WORK / 'results/summary/contribution_decomposition.csv'))

In [ ]:
# Export compact evidence: CSV/JSON/log files, without generated images or large checkpoints.
EVIDENCE = Path('/kaggle/working/transferred_teacher_integrity_evidence')
if EVIDENCE.exists():
    shutil.rmtree(EVIDENCE)
EVIDENCE.mkdir()
for path in WORK.rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.csv', '.json', '.md'}:
        rel = path.relative_to(WORK)
        target = EVIDENCE / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
shutil.make_archive('/kaggle/working/transferred_teacher_integrity_evidence', 'zip', EVIDENCE)
print('/kaggle/working/transferred_teacher_integrity_evidence.zip')
print('Best checkpoints remain under:', WORK / 'results/checkpoints')